# **Structured Outputs & Output Parsers: LangChain and Native OpenAI SDK Integration**

This notebook provides a hands-on exploration and comparison of various techniques for extracting **structured, type-safe data** from Large Language Models (LLMs). Using a consistent **`MovieRecommendation`** Pydantic schema (`title`, `genre`, and `release_year`), the notebook demonstrates how to handle output parsing through both high-level LangChain utilities and direct native API calls.

#### **Key Sections Covered in the Notebook:**

* **Schema Definition**: Establishes a reusable `MovieRecommendation` Pydantic model to maintain a uniform data structure across all examples.
* **LangChain Parsers (Simulated Outputs)**:
* **`JsonOutputParser`**: Parses LLM output into a dictionary using Pydantic schema guidance.
* **`StructuredOutputParser`**: Uses a collection of `ResponseSchema` rules to parse markdown JSON code blocks into a dictionary.
* **`PydanticToolsParser`**: Processes tool call messages (`AIMessage`) to return fully instantiated Pydantic objects.
* **`PydanticOutputParser`**: Validates and transforms text output directly into a strongly typed Pydantic object accessible via dot-notation.


* **Live OpenAI SDK Integrations**:
* **JSON Output Parsing with OpenAI (`run_openai_with_json_output_parser`)**: Combines the native OpenAI chat completions endpoint (`client.chat.completions.create`) with LangChain's `JsonOutputParser` instructions to process live responses.


* **Tool/Function Calling with OpenAI (`run_openai_tool_calling_parse`)**: Leverages OpenAI's native tool-calling architecture and argument parsing to extract structured JSON data reliably without external framework dependencies.

In [1]:
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import StructuredOutputParser, ResponseSchema, PydanticToolsParser, PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

from openai import OpenAI
from dotenv import load_dotenv
import json

/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
load_dotenv()

False

## Shared Pydantic Data Schema

Define a reusable `MovieRecommendation` Pydantic model representing the expected target output structure:
* `title` (*string*): The title of the movie.
* `genre` (*string*): The primary genre of the movie.
* `release_year` (*integer*): The release year of the movie.

In [3]:
class MovieRecommendation(BaseModel):
    title: str = Field(description="The title of the movie")
    genre: str = Field(description="The primary genre of the movie")
    release_year: int = Field(description="The year the movie was released")

### `JsonOutputParser` Demonstration

This function demonstrates how **`JsonOutputParser`** uses a Pydantic schema to generate formatting instructions for an LLM. It parses a simulated JSON string output into a standard Python dictionary (`dict`).

In [4]:
# ==========================================
# 1. JsonOutputParser Example
# ==========================================
def demonstrate_json_parser():
    print("--- 1. JsonOutputParser ---")
    parser = JsonOutputParser(pydantic_object=MovieRecommendation)
    
    format_instructions = parser.get_format_instructions()
    print("Format Instructions:\n", format_instructions)
    
    simulated_llm_output = '{"title": "Inception", "genre": "Sci-Fi", "release_year": 2010}'
    parsed_result = parser.parse(simulated_llm_output)
    
    print("Parsed Output:", parsed_result)
    print("Type:", type(parsed_result))
    print("-" * 40)

In [5]:
demonstrate_json_parser()

--- 1. JsonOutputParser ---
Format Instructions:
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "The title of the movie", "title": "Title", "type": "string"}, "genre": {"description": "The primary genre of the movie", "title": "Genre", "type": "string"}, "release_year": {"description": "The year the movie was released", "title": "Release Year", "type": "integer"}}, "required": ["title", "genre", "release_year"]}
```
Parsed Output: {'title': 'Inception', 'genre': 'Sci-Fi', 'release_year': 2010}
Type: <class 'dict'>
----------------------------------

## `StructuredOutputParser` (LangChain)

Demonstrates `StructuredOutputParser` using explicit `ResponseSchema` objects. This parser instructs the model to wrap its JSON response inside a Markdown code block (```json) and parses it back into a Python dictionary (`dict`).

In [6]:
# ==========================================
# 2. StructuredOutputParser Example
# ==========================================
def demonstrate_structured_parser():
    print("--- 2. StructuredOutputParser ---")
    
    # Aligned fields mirroring MovieRecommendation
    response_schemas = [
        ResponseSchema(name="title", description="The title of the movie"),
        ResponseSchema(name="genre", description="The primary genre of the movie"),
        ResponseSchema(name="release_year", description="The year the movie was released (as a number)")
    ]
    
    parser = StructuredOutputParser.from_response_schemas(response_schemas)
    format_instructions = parser.get_format_instructions()
    print("Format Instructions:\n", format_instructions)
    
    simulated_llm_output = """```json
{
    "title": "Interstellar",
    "genre": "Sci-Fi",
    "release_year": 2014
}
```"""
    
    parsed_result = parser.parse(simulated_llm_output)
    print("Parsed Output:", parsed_result)
    print("Type:", type(parsed_result))
    print("-" * 40)

In [7]:
demonstrate_structured_parser()

--- 2. StructuredOutputParser ---
Format Instructions:
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"title": string  // The title of the movie
	"genre": string  // The primary genre of the movie
	"release_year": string  // The year the movie was released (as a number)
}
```
Parsed Output: {'title': 'Interstellar', 'genre': 'Sci-Fi', 'release_year': 2014}
Type: <class 'dict'>
----------------------------------------


## `PydanticToolsParser` (LangChain)

Demonstrates parsing tool calls inside an `AIMessage` using `PydanticToolsParser`. By passing the `MovieRecommendation` Pydantic class directly into the parser, tool call arguments are parsed and instantiated directly into Pydantic model objects.

In [8]:
# ==========================================
# 3. Tool Output Parser Example (PydanticToolsParser)
# ==========================================
def demonstrate_tool_output_parser():
    print("--- 3. Tool Output Parser (PydanticToolsParser) ---")
    
    # Pass the MovieRecommendation Pydantic class directly as the tool schema
    parser = PydanticToolsParser(tools=[MovieRecommendation])
    
    # Simulated AIMessage where the LLM invokes the 'MovieRecommendation' tool
    simulated_message = AIMessage(
        content="",
        tool_calls=[
            {
                "name": "MovieRecommendation",
                "args": {
                    "title": "The Matrix",
                    "genre": "Action / Sci-Fi",
                    "release_year": 1999
                },
                "id": "call_movie_rec_123"
            }
        ]
    )
    
    # Use .invoke() to parse the AIMessage into a MovieRecommendation instance
    parsed_result = parser.invoke(simulated_message)
    
    print("Parsed Tool Call Object:", parsed_result[0])
    print("Movie Title:", parsed_result[0].title)
    print("Movie Genre:", parsed_result[0].genre)
    print("Release Year:", parsed_result[0].release_year)
    print("-" * 40)

In [9]:
demonstrate_tool_output_parser()

--- 3. Tool Output Parser (PydanticToolsParser) ---
Parsed Tool Call Object: title='The Matrix' genre='Action / Sci-Fi' release_year=1999
Movie Title: The Matrix
Movie Genre: Action / Sci-Fi
Release Year: 1999
----------------------------------------


## `PydanticOutputParser` (LangChain)

Demonstrates `PydanticOutputParser`. Similar to `JsonOutputParser`, but instead of returning a dictionary, it parses the JSON string into an actual **Pydantic object instance**, enabling attribute access via dot-notation (e.g., `parsed_result.title`).

In [10]:
# ==========================================
# 4. PydanticOutputParser Example (Returns a Pydantic Object)
# ==========================================
def demonstrate_pydantic_output_parser():
    print("--- 3. PydanticOutputParser ---")
    
    parser = PydanticOutputParser(pydantic_object=MovieRecommendation)
    
    format_instructions = parser.get_format_instructions()
    print("Format Instructions:\n", format_instructions)
    
    simulated_llm_output = '{"title": "The Dark Knight", "genre": "Action", "release_year": 2008}'
    parsed_result = parser.parse(simulated_llm_output)
    
    print("Parsed Output:", parsed_result)
    print("Type:", type(parsed_result))
    print("Accessing attribute via dot notation:", parsed_result.title)
    print("-" * 40)

In [11]:
 demonstrate_pydantic_output_parser()

--- 3. PydanticOutputParser ---
Format Instructions:
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "The title of the movie", "title": "Title", "type": "string"}, "genre": {"description": "The primary genre of the movie", "title": "Genre", "type": "string"}, "release_year": {"description": "The year the movie was released", "title": "Release Year", "type": "integer"}}, "required": ["title", "genre", "release_year"]}
```
Parsed Output: title='The Dark Knight' genre='Action' release_year=2008
Type: <class '__main__.MovieRecommendation'>
Accessing att

## Live LLM Call with `JsonOutputParser` & OpenAI SDK

Integrates LangChain's `JsonOutputParser` with a live OpenAI SDK call (`client.chat.completions.create`). 
* Format instructions generated by the parser are injected directly into the user prompt.
* The API enforces `json_object` format.
* The raw text output from OpenAI is passed into `parser.parse()` to get a structured Python dictionary.

In [12]:
def run_openai_with_json_output_parser(query: str):
    # Initialize the OpenAI client and LangChain's JsonOutputParser
    client = OpenAI()
    parser = JsonOutputParser(pydantic_object=MovieRecommendation)
    
    # 2. Get the formatting instructions generated by the parser
    format_instructions = parser.get_format_instructions()
    
    # 3. Include format instructions inside your prompt/messages
    prompt = f"{query}\n\n{format_instructions}"

    # 4. Make the standard OpenAI chat completions call
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful movie recommendation assistant."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )

    # Extract the raw string content from the response
    raw_content = response.choices[0].message.content
    print("Raw LLM Response:\n", raw_content)
    print("-" * 40)

    # 5. Use LangChain's parser to parse the raw text string
    parsed_result = parser.parse(raw_content)

    print("Parsed Result via JsonOutputParser:")
    print(parsed_result)
    print("Python Type:", type(parsed_result))
    print("Movie Title:", parsed_result.get("title"))
    print("Genre:", parsed_result.get("genre"))
    print("Release Year:", parsed_result.get("release_year"))

In [13]:
run_openai_with_json_output_parser("Recommend a critically acclaimed mind-bending sci-fi movie directed by Christopher Nolan.")

Raw LLM Response:
 {
  "title": "Inception",
  "genre": "Science Fiction",
  "release_year": 2010
}
----------------------------------------
Parsed Result via JsonOutputParser:
{'title': 'Inception', 'genre': 'Science Fiction', 'release_year': 2010}
Python Type: <class 'dict'>
Movie Title: Inception
Genre: Science Fiction
Release Year: 2010


## Live Structured Output using Native OpenAI Tool Calling

Demonstrates structured data extraction using native OpenAI tool calling without LangChain parser dependencies.
* Defines the JSON schema as an OpenAI tool (`MovieRecommendation`).
* Forces the model to invoke the tool via `tool_choice`.
* Extracts and parses the function arguments payload directly using Python's built-in `json.loads()`.

In [14]:
def run_openai_tool_calling_parse(query: str):
    # Initialize the OpenAI client
    client = OpenAI()

    # 1. Define the tool schema (similar to a Pydantic structure)
    tools = [
            {
                "type": "function",
                "function": {
                    "name": "MovieRecommendation",
                    "description": "Provide a movie recommendation with its title, genre, and release year.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "title": {
                                "type": "string", 
                                "description": "The title of the movie"
                            },
                            "genre": {
                                "type": "string", 
                                "description": "The primary genre of the movie"
                            },
                            "release_year": {
                                "type": "integer", 
                                "description": "The year the movie was released"
                            }
                        },
                        "required": ["title", "genre", "release_year"],
                        "additionalProperties": False
                    }
                }
            }
    ]

    # 2. Make the chat completion call with tools enabled and forced
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. You must use the provided tool to respond."},
            {"role": "user", "content": query}
        ],
        tools=tools,
        tool_choice={"type": "function", "function": {"name": "MovieRecommendation"}} # Forces the model to use this tool
    )

    # 3. Extract the tool call arguments payload
    message = response.choices[0].message
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        arguments_string = tool_call.function.arguments
        
        print("Raw Tool Call Arguments String:\n", arguments_string)
        print("-" * 40)

        # 4. Parse the arguments JSON string into a Python dictionary
        parsed_result = json.loads(arguments_string)

        print("Parsed Python Dictionary:")
        print(parsed_result)
        print("Python Type:", type(parsed_result))
        print("Movie Title:", parsed_result.get("title"))
        print("Release Year:", parsed_result.get("release_year"))

In [15]:
run_openai_tool_calling_parse("Recommend a critically acclaimed mind-bending sci-fi movie directed by Christopher Nolan.")

Raw Tool Call Arguments String:
 {"title":"Inception","genre":"Science Fiction","release_year":2010}
----------------------------------------
Parsed Python Dictionary:
{'title': 'Inception', 'genre': 'Science Fiction', 'release_year': 2010}
Python Type: <class 'dict'>
Movie Title: Inception
Release Year: 2010
